# Decoder-Only Transformer (v3 -- Multi-Head Attention)

Same tiny GPT-style decoder as v1, but the single attention head is replaced by
**multi-head attention**: `vocab_size = 12`, `d_model = 8`, `num_heads = 2`, one
decoder block.

## Why more than one head?

One head produces *one* attention pattern per position -- a single opinion about
"who should I look at". With `num_heads = 2` the model gets two independent
opinions computed in parallel, then concatenates them and mixes them with an
output projection `Wo`.

The trick is that heads are **not** extra parameters: `d_model = 8` is *split*
into `num_heads = 2` slices of `d_k = d_model // num_heads = 4`. Same parameter
count as v1 (plus `Wo`), more expressive attention.

```
Wq(x) -> (4, 7, 8)   one big projection
         reshape     (4, 7, 2, 4)   split the width into 2 heads of 4
         transpose   (4, 2, 7, 4)   heads become a batch dimension
                     ^^^^^^^^^^^^   attention now runs on 2 independent (7, 4) blocks
```


## Shapes at every step

| Stage | Tensor | Shape |
|---|---|---|
| input | `X` | `(4, 7)` |
| after embedding | word vectors | `(4, 7, 8)` |
| after position encoding | positioned vectors | `(4, 7, 8)` |
| after `Wq` / `Wk` / `Wv` | projections | `(4, 7, 8)` |
| after `split_heads` | per-head Q/K/V | `(4, 2, 7, 4)` -- batch x head x seq x d_k |
| attention scores | `Q @ K.T / sqrt(4)` | `(4, 2, 7, 7)` -- one seq x seq map **per head** |
| after `merge_heads` | concatenated heads | `(4, 7, 8)` |
| after `Wo` | mixed heads | `(4, 7, 8)` |
| after decoder block | contextualized vectors | `(4, 7, 8)` |
| after `fc` | logits | `(4, 7, 12)` -- seq x vocab |

> `seq = 7` because each sentence is 8 tokens and the input drops the last one
> (`sentence[:-1]`). The trailing dims tell the story: **7** is *position*, **8** is
> the *model width* `d_model`, **4** is *one head's width* `d_k`, and **12** is the
> *vocabulary* size.
>
> Note the scaling factor is `sqrt(d_k)`, not `sqrt(d_model)` -- each head only
> sums over `d_k` dimensions, so that is the variance it needs to undo.

In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
# =====================================================
# Vocabulary
# =====================================================
token_to_id = {
    "<EOS>": 0,
    "<PAD>": 1,
    "x": 2,
    "y": 3,
    "xy": 4,
    "yx": 5,
    "wife": 6,
    "hubby": 7,
    "daughter": 8,
    "son": 9,
    "is": 10,
    "and": 11,

}

id_to_token = {v: k for k, v in token_to_id.items()}

vocab_size = len(token_to_id)
d_model = 8
num_heads = 2
max_len = 12

In [5]:
# =====================================================
# Training Sentences
#
# The first <EOS> separates the prompt from the answer.
# The trailing <EOS> is the real end of the sequence -- without
# it, "awesome" is never seen as an input, so the model has no
# idea what follows it and generation can't learn to stop.
# =====================================================
sentences = [
    ["y", "is", "x", "<EOS>", "wife", "<EOS>", "<PAD>", "<PAD>"],
    ["x", "is", "y", "<EOS>", "hubby", "<EOS>", "<PAD>", "<PAD>"],
    ["xy", "is", "<EOS>", "x", "and", "y", "son", "<EOS>"],
    ["yx", "is", "<EOS>", "x", "and", "y", "daughter", "<EOS>"],
]

In [6]:
# =====================================================
# Build Training Data
# Input  = sentence[:-1]
# Target = sentence[1:]
# =====================================================
X = []
Y = []

for sentence in sentences:
    ids = [token_to_id[word] for word in sentence]
    X.append(ids[:-1])
    Y.append(ids[1:])

X = torch.tensor(X)
Y = torch.tensor(Y)

print("Input")
print(X)

print("\nTarget")
print(Y)

Input
tensor([[ 3, 10,  2,  0,  6,  0,  1],
        [ 2, 10,  3,  0,  7,  0,  1],
        [ 4, 10,  0,  2, 11,  3,  9],
        [ 5, 10,  0,  2, 11,  3,  8]])

Target
tensor([[10,  2,  0,  6,  0,  1,  1],
        [10,  3,  0,  7,  0,  1,  1],
        [10,  0,  2, 11,  3,  9,  0],
        [10,  0,  2, 11,  3,  8,  0]])


In [7]:
# =====================================================
# Positional Encoding
# =====================================================
class PositionEncoding(nn.Module):

    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1)

        embedding_index = torch.arange(0, d_model, 2)

        div_term = torch.exp(
            torch.arange(0, d_model, 2)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)


        self.register_buffer("pe", pe)

    def forward(self, word_embeddings):
        seq_len = word_embeddings.size(-2)
        return word_embeddings + self.pe[:seq_len]

In [8]:
# =====================================================
# Multi Head Masked Attention
#
# d_model is SPLIT across heads, not duplicated:
#     d_k = d_model // num_heads   ->  8 // 2 = 4
#
# Wq/Wk/Wv stay (d_model, d_model). Every head's slice of the
# output lives in a different chunk of those 8 columns, so one
# matmul computes all heads at once. The reshape below is what
# actually separates them.
# =====================================================
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)

        # Mixes the concatenated heads back together. Without this
        # the heads never talk to each other.
        self.Wo = nn.Linear(d_model, d_model, bias=False)

    def split_heads(self, x):
        # (batch, seq, d_model) -> (batch, num_heads, seq, d_k)
        batch, seq_len, _ = x.size()

        x = x.view(batch, seq_len, self.num_heads, self.d_k)

        # heads move next to batch so attention treats them as
        # independent parallel problems
        return x.transpose(1, 2)

    def merge_heads(self, x):
        # (batch, num_heads, seq, d_k) -> (batch, seq, d_model)
        batch, _, seq_len, _ = x.size()

        x = x.transpose(1, 2).contiguous()

        return x.view(batch, seq_len, self.num_heads * self.d_k)

    def forward(self, x, return_attention=False):

        Q = self.split_heads(self.Wq(x))
        K = self.split_heads(self.Wk(x))
        V = self.split_heads(self.Wv(x))

        # (batch, heads, seq, d_k) @ (batch, heads, d_k, seq)
        #   -> (batch, heads, seq, seq)
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # scale by the width of ONE head, not d_model
        scores = scores / math.sqrt(self.d_k)

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        ).bool()

        # (seq, seq) broadcasts over batch and heads --
        # every head gets the same causal mask
        scores = scores.masked_fill(mask, -1e9)

        attention = F.softmax(scores, dim=-1)

        output = torch.matmul(attention, V)

        output = self.merge_heads(output)

        output = self.Wo(output)

        if return_attention:
            return output, attention

        return output

In [9]:
# =====================================================
# Decoder Block
# =====================================================
class DecoderBlock(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)

        self.norm1 = nn.LayerNorm(d_model)

        self.ff = nn.Sequential(
            nn.Linear(d_model, 8),
            nn.ReLU(),
            nn.Linear(8, d_model),
        )

        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):

        attn = self.attention(x)

        x = self.norm1(x + attn)

        ff = self.ff(x)

        x = self.norm2(x + ff)

        return x

In [10]:
# =====================================================
# Decoder Only Transformer
# =====================================================
class DecoderOnlyTransformer(nn.Module):

    def __init__(self, vocab_size, d_model, num_heads, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position = PositionEncoding(d_model, max_len)
        self.decoder = DecoderBlock(d_model, num_heads)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        x = self.position(x)
        x = self.decoder(x)
        logits = self.fc(x)
        return logits

In [11]:
# =====================================================
# Create Model
# =====================================================
model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    max_len=max_len,
)

criterion = nn.CrossEntropyLoss(ignore_index=token_to_id["<PAD>"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
)

In [12]:
# =====================================================
# Shape Check
# Walk one batch through the pieces and print what comes out.
# =====================================================
with torch.no_grad():

    emb = model.embedding(X)
    pos = model.position(emb)

    attn_layer = model.decoder.attention

    q = attn_layer.Wq(pos)
    q_heads = attn_layer.split_heads(q)

    out, weights = attn_layer(pos, return_attention=True)

print("X                 ", tuple(X.shape))
print("after embedding   ", tuple(emb.shape))
print("after position    ", tuple(pos.shape))
print("after Wq          ", tuple(q.shape))
print("after split_heads ", tuple(q_heads.shape), " <- batch, heads, seq, d_k")
print("attention weights ", tuple(weights.shape), " <- one seq x seq map per head")
print("attention output  ", tuple(out.shape))
print("logits            ", tuple(model(X).shape))

X                  (4, 7)
after embedding    (4, 7, 8)
after position     (4, 7, 8)
after Wq           (4, 7, 8)
after split_heads  (4, 2, 7, 4)  <- batch, heads, seq, d_k
attention weights  (4, 2, 7, 7)  <- one seq x seq map per head
attention output   (4, 7, 8)
logits             (4, 7, 12)


In [13]:
# =====================================================
# Training
# =====================================================
epochs = 1000

for epoch in range(epochs):

    optimizer.zero_grad()

    logits = model(X)

    loss = criterion(
        logits.reshape(-1, vocab_size),
        Y.reshape(-1),
    )

    loss.backward()

    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}  Loss = {loss.item():.4f}")

Epoch    0  Loss = 2.6359
Epoch  100  Loss = 0.0212
Epoch  200  Loss = 0.0039
Epoch  300  Loss = 0.0019
Epoch  400  Loss = 0.0011
Epoch  500  Loss = 0.0007
Epoch  600  Loss = 0.0005
Epoch  700  Loss = 0.0004
Epoch  800  Loss = 0.0303
Epoch  900  Loss = 0.0008


In [14]:
# =====================================================
# Predictions
# =====================================================
print("\nPredictions")

model.eval()

with torch.no_grad():

    logits = model(X)

    predictions = logits.argmax(dim=-1)

print(predictions)

print("\nDecoded Predictions")

for sentence in predictions:

    words = [id_to_token[token.item()] for token in sentence]

    print(words)


Predictions
tensor([[10,  2,  0,  6,  0,  6,  8],
        [10,  3,  0,  7,  0,  7,  8],
        [10,  0,  2, 11,  3,  9,  0],
        [10,  0,  2, 11,  3,  8,  0]])

Decoded Predictions
['is', 'x', '<EOS>', 'wife', '<EOS>', 'wife', 'daughter']
['is', 'y', '<EOS>', 'hubby', '<EOS>', 'hubby', 'daughter']
['is', '<EOS>', 'x', 'and', 'y', 'son', '<EOS>']
['is', '<EOS>', 'x', 'and', 'y', 'daughter', '<EOS>']


In [15]:
# =====================================================
# What did each head learn?
#
# Row i = what position i attends to. The upper triangle is
# always 0 because of the causal mask. Compare head 0 vs head 1
# on the same sentence -- they should NOT be identical, that is
# the whole point of having more than one.
# =====================================================
sentence_index = 3

tokens = [id_to_token[t.item()] for t in X[sentence_index]]

with torch.no_grad():
    x = model.position(model.embedding(X))
    _, weights = model.decoder.attention(x, return_attention=True)

print("Sentence:", tokens)

for head in range(num_heads):

    print(f"\nHead {head}")

    header = "".join(f"{t:>10}" for t in tokens)
    print(f"{'':>10}{header}")

    for i, row_token in enumerate(tokens):

        row = weights[sentence_index, head, i]

        cells = "".join(f"{value:>10.2f}" for value in row)

        print(f"{row_token:>10}{cells}")

Sentence: ['yx', 'is', '<EOS>', 'x', 'and', 'y', 'daughter']

Head 0
                  yx        is     <EOS>         x       and         y  daughter
        yx      1.00      0.00      0.00      0.00      0.00      0.00      0.00
        is      1.00      0.00      0.00      0.00      0.00      0.00      0.00
     <EOS>      0.08      0.16      0.76      0.00      0.00      0.00      0.00
         x      0.01      0.98      0.00      0.00      0.00      0.00      0.00
       and      0.15      0.07      0.00      0.00      0.77      0.00      0.00
         y      0.00      0.00      0.99      0.01      0.00      0.00      0.00
  daughter      0.01      0.00      0.09      0.01      0.00      0.86      0.03

Head 1
                  yx        is     <EOS>         x       and         y  daughter
        yx      1.00      0.00      0.00      0.00      0.00      0.00      0.00
        is      0.37      0.63      0.00      0.00      0.00      0.00      0.00
     <EOS>      0.00      0.00  

In [16]:
# =====================================================
# Autoregressive Generation
#
# <EOS> here is the prompt/response separator, not the end
# of the sequence -- the answer comes after it. So the first
# <EOS> is passed over and we stop on the second one.
# =====================================================
print("\nGeneration")

prompt = ["yx"]

generated = prompt.copy()

seen_eos = False

for _ in range(max_len - len(generated)):

    ids = [token_to_id[word] for word in generated]

    x = torch.tensor([ids])

    with torch.no_grad():
        logits = model(x)

    next_token = logits[0, -1].argmax().item()

    next_word = id_to_token[next_token]

    generated.append(next_word)

    if next_word == "<EOS>":

        if seen_eos:
            break

        seen_eos = True

output = " ".join(word for word in generated if word != "<EOS>")
print("Prompt :", prompt)
print("Output :", output)


Generation
Prompt : ['yx']
Output : yx is x and y daughter
